In [1]:
# ManySig 跨日期 SNR循环 6TX
from joblib import load
import pandas as pd
import numpy as np
import os
from data_utilities import *
import cv2
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import gc
from tqdm.auto import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import seaborn as sns
from datetime import datetime

dataset_name = 'ManySig'
dataset_path='../ManySig.pkl/'

compact_dataset = load_compact_pkl_dataset(dataset_path,dataset_name)

print("数据集发射机数量：",len(compact_dataset['tx_list']),"具体为：",compact_dataset['tx_list'])
print("数据集接收机数量：",len(compact_dataset['rx_list']),"具体为：",compact_dataset['rx_list'])
print("数据集采集天数：",len(compact_dataset['capture_date_list']),"具体为：",compact_dataset['capture_date_list'])


tx_list = compact_dataset['tx_list']
rx_list = compact_dataset['rx_list']
equalized = 0
capture_date_list = compact_dataset['capture_date_list']


n_tx = len(tx_list)
n_rx = len(rx_list)
print(n_tx,n_rx)


train_dates = ['2021_03_15']  # 设定训练日期
test_dates  = ['2021_03_01']  # 设定测试日期
X_train, y_train, X_test, y_test = preprocess_dataset_for_classification_cross_date(
    compact_dataset, tx_list, rx_list, train_dates, test_dates, max_sig=None, equalized = equalized)

print("训练集所选日期：",train_dates, "测试集所选日期：", test_dates)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test  shape:", X_test.shape)
print("y_test  shape:", y_test.shape)

import os
import numpy as np
import torch
torch._dynamo.disable()
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.auto import tqdm
import pywt

# ====================== 参数设置 ======================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 数据处理参数
USE_LOG = True
WAVELET = 'db6'
WAVELET_LEVEL = 6
FS = 20e6
FC = 2.4e9
SNR_DB = 20           # None 或具体数值
VELOCITY_KMH = 120     # None 或具体数值
ADD_NOISE = True
ADD_DOPPLER = True

# 训练参数
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5
PATIENCE = 5

# 保存路径
SAVE_ROOT = "./training_results"
os.makedirs(SAVE_ROOT, exist_ok=True)

# ====================== 数据处理函数 ======================
def compute_doppler_shift(v_kmh, fc_hz):
    if not v_kmh: return 0
    c = 3e8
    v_mps = v_kmh / 3.6
    return fc_hz * v_mps / c

def add_complex_awgn(signal, snr_db):
    """
    为复数信号添加AWGN噪声
    """
    # 计算信号功率
    signal_power = np.mean(np.abs(signal) ** 2)
    
    # 计算噪声功率
    snr_linear = 10 ** (snr_db / 10)
    noise_power = signal_power / snr_linear
    
    # 生成复数噪声（实部和虚部独立，各占一半功率）
    noise_std = np.sqrt(noise_power / 2)
    noise_real = np.random.normal(0, noise_std, signal.shape)
    noise_imag = np.random.normal(0, noise_std, signal.shape)
    noise = noise_real + 1j * noise_imag
    
    return signal + noise

def apply_doppler_shift(signal, fd_hz, fs_hz):
    if fd_hz is None or fd_hz == 0:
        return signal
    t = np.arange(len(signal)) / fs_hz
    return signal * np.exp(1j * 2 * np.pi * fd_hz * t)

def process_signal_led_rff(sig_complex, use_log=False, wavelet='db6', level=6):
    # --------------------
    # 1. 先做 FFT
    # --------------------
    freq_sig = np.fft.fft(sig_complex)
    amp = np.abs(freq_sig)   # 幅度谱
    # 可选只取前半段（对称）
    amp = amp[:len(amp)//2]

    # --------------------
    # 2. 可选 log 处理
    # --------------------
    if use_log:
        amp = np.log(amp + 1e-8)

    # --------------------
    # 3. 小波分解 + 低频置零 + 重构
    # --------------------
    coeffs = pywt.wavedec(amp, wavelet, level=level)
    coeffs[0] = np.zeros_like(coeffs[0])
    rec = pywt.waverec(coeffs, wavelet)
    rec = rec[:len(amp)]

    # --------------------
    # 4. 归一化
    # --------------------
    mu, sigma = rec.mean(), rec.std()
    if sigma < 1e-8:
        feat = (rec - mu).astype(np.float32)
    else:
        feat = ((rec - mu) / (sigma + 1e-8)).astype(np.float32)
    return feat


def preprocess_iq_dataset_led_rff(data_real_imag, snr_db=SNR_DB, velocity_kmh=VELOCITY_KMH,
                                  fc_hz=FC, fs_hz=FS, use_log=USE_LOG, wavelet=WAVELET,
                                  level=WAVELET_LEVEL, add_noise=ADD_NOISE, add_doppler=ADD_DOPPLER):
    num_samples, sig_len, _ = data_real_imag.shape
    processed_feats = []

    data_complex = data_real_imag[...,0] + 1j*data_real_imag[...,1]
    fd_hz = compute_doppler_shift(velocity_kmh, fc_hz) if add_doppler else None

    for i in range(num_samples):
        sig = data_complex[i]
        if add_doppler: sig = apply_doppler_shift(sig, fd_hz, fs_hz)
        if add_noise: sig = add_complex_awgn(sig, snr_db)
        feat = process_signal_led_rff(sig, use_log=use_log, wavelet=wavelet, level=level)
        processed_feats.append(feat)

    processed_feats = np.stack(processed_feats, axis=0)
    return torch.tensor(processed_feats, dtype=torch.float32)[:, None, :]


# ====================== InceptionTime 模型 ======================
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        bottleneck_channels = max(1, out_channels // 4)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.conv1 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=10, padding=5)
        self.conv2 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=20, padding=10)
        self.conv3 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=40, padding=20)
        self.maxpool = nn.MaxPool1d(3, stride=1, padding=1)
        self.convpool = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(4*out_channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        x_b = self.bottleneck(x)
        c1 = self.conv1(x_b)
        c2 = self.conv2(x_b)
        c3 = self.conv3(x_b)
        c4 = self.convpool(self.maxpool(x_b))
        min_len = min(c1.shape[-1], c2.shape[-1], c3.shape[-1], c4.shape[-1])
        c1=c1[...,:min_len]; c2=c2[...,:min_len]; c3=c3[...,:min_len]; c4=c4[...,:min_len]
        out = torch.cat([c1,c2,c3,c4], dim=1)
        return self.relu(self.bn(out))

class InceptionTime(nn.Module):
    def __init__(self, num_classes, in_channels=1, channels=32):
        super().__init__()
        self.b1 = InceptionBlock(in_channels, channels)
        self.b2 = InceptionBlock(4*channels, channels)
        self.b3 = InceptionBlock(4*channels, channels)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4*channels, num_classes)
    def forward(self, x):
        if x.shape[-1] % 2 == 1: x = x[...,:-1]
        x = self.b1(x); x = self.b2(x); x = self.b3(x)
        x = self.gap(x).squeeze(-1)
        return self.fc(x)

# ====================== 工具函数 ======================
def evaluate_model(model, dataloader, device, num_classes):
    model.eval()
    correct, total = 0, 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            _, p = torch.max(out, 1)
            correct += (p == yb).sum().item()
            total += yb.size(0)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(p.cpu().numpy())
    acc = 100.0 * correct / total
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    return acc, cm

def plot_confusion_matrix(cm, classes, fold, save_folder, dataset_type='Test'):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{dataset_type} Confusion Matrix Fold{fold}')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.savefig(os.path.join(save_folder,f'{dataset_type.lower()}_cm_fold{fold}.png'))
    plt.close()

def plot_curves(train_losses, val_losses, train_acc, val_acc, fold, save_folder):
    plt.figure(); plt.plot(train_losses,label='Train Loss'); plt.plot(val_losses,label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Fold {fold} Loss'); plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(save_folder,f'loss_fold{fold}.png')); plt.close()
    plt.figure(); plt.plot(train_acc,label='Train Acc'); plt.plot(val_acc,label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title(f'Fold {fold} Accuracy'); plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(save_folder,f'acc_fold{fold}.png')); plt.close()

# ====================== KFold 训练（带参数保存 + 混淆矩阵 + 曲线 + 平均梯度范数） ======================
def train_kfold(X_train, y_train, X_test, y_test, num_classes, device=DEVICE, 
                     snr_db=SNR_DB, velocity_kmh=VELOCITY_KMH, fc_hz=FC, fs_hz=FS,
                     wavelet=WAVELET, wavelet_level=WAVELET_LEVEL,
                     batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                     n_splits=N_SPLITS, patience=PATIENCE):

    fd_hz = compute_doppler_shift(velocity_kmh, fc_hz)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    script_name='wisig_LED'
    save_dir = f"{timestamp}_{script_name}_SNR{snr_db}dB_fd{int(fd_hz)}_classes_{num_classes}_CNN"
    save_folder = os.path.join(SAVE_ROOT, save_dir)
    os.makedirs(save_folder, exist_ok=True)
    results_file = os.path.join(save_folder,"results.txt")

    # 保存实验参数
    with open(results_file,'w') as f:
        f.write("=== Experiment Parameters ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Device: {device}\n")
        f.write(f"SNR_dB: {snr_db}\nDoppler_fd: {fd_hz:.2f} Hz\nFS: {fs_hz}\nFC: {fc_hz}\n")
        f.write(f"Wavelet: {wavelet}, Level: {wavelet_level}\n")
        f.write(f"Batch size: {batch_size}, Epochs: {epochs}, LR: {lr}, Weight decay: {weight_decay}\n")
        f.write(f"K-Fold: {n_splits}, Patience: {patience}, Num classes: {num_classes}\n")
        f.write("============================\n\n")

    full_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(full_dataset))

    val_scores, test_scores = [], []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(indices)):
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        tr_sub, va_sub = Subset(full_dataset, tr_idx), Subset(full_dataset, va_idx)
        tr_loader = DataLoader(tr_sub, batch_size=batch_size, shuffle=True)
        va_loader = DataLoader(va_sub, batch_size=batch_size, shuffle=False)

        model = InceptionTime(num_classes=num_classes, in_channels=X_train.shape[1]).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

        best_val, best_wts, patience_cnt = 0.0, None, 0
        train_losses, val_losses, train_acc_list, val_acc_list, avg_grad_list = [], [], [], [], []

        for epoch in range(epochs):
            model.train(); running_loss, correct, total = 0.0,0,0
            total_grad = 0.0; count_grad = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                out = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                _, p = torch.max(out,1)
                correct += (p==yb).sum().item()
                total += yb.size(0)
                # 平均梯度范数
                grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
                if grad_norms:
                    total_grad += np.mean(grad_norms)
                    count_grad += 1
            avg_grad = total_grad / max(count_grad,1)
            avg_grad_list.append(avg_grad)

            train_loss = running_loss / len(tr_loader)
            train_acc = 100.0*correct/total
            train_losses.append(train_loss)
            train_acc_list.append(train_acc)

            # Validation
            model.eval(); vloss,vcorrect,vtotal=0.0,0,0
            with torch.no_grad():
                all_labels, all_preds = [], []
                for xb,yb in va_loader:
                    xb,yb = xb.to(device), yb.to(device)
                    out = model(xb)
                    loss = criterion(out,yb)
                    vloss += loss.item()
                    _,p = torch.max(out,1)
                    vcorrect += (p==yb).sum().item()
                    vtotal += yb.size(0)
                    all_labels.extend(yb.cpu().numpy())
                    all_preds.extend(p.cpu().numpy())
            val_loss = vloss / len(va_loader)
            val_acc = 100.0*vcorrect/vtotal
            val_losses.append(val_loss)
            val_acc_list.append(val_acc)
            val_cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
            np.save(os.path.join(save_folder,f'val_cm_fold{fold+1}.npy'), val_cm)

            print(f"Epoch {epoch+1}/{epochs} | TrainAcc={train_acc:.2f}% | ValAcc={val_acc:.2f}% | "
                  f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | AvgGrad={avg_grad:.4f}")
            with open(results_file,'a') as f:
                f.write(f"Fold{fold+1} Epoch{epoch+1} | TrainAcc={train_acc:.2f}% | ValAcc={val_acc:.2f}% | "
                        f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | AvgGrad={avg_grad:.4f}\n")

            # Early stopping
            if val_acc > best_val + 0.01:
                best_val = val_acc
                best_wts = model.state_dict()
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience:
                    print("Early stopping.")
                    break
            scheduler.step()

        if best_wts is not None:
            model.load_state_dict(best_wts)

        # Train/Val confusion matrices
        train_acc, train_cm = evaluate_model(model, tr_loader, device, num_classes)
        np.save(os.path.join(save_folder,f'train_cm_fold{fold+1}.npy'), train_cm)
        plot_confusion_matrix(train_cm, classes=list(range(num_classes)), fold=fold+1, save_folder=save_folder, dataset_type='Train')

        val_acc, val_cm = evaluate_model(model, va_loader, device, num_classes)
        np.save(os.path.join(save_folder,f'val_cm_fold{fold+1}.npy'), val_cm)
        plot_confusion_matrix(val_cm, classes=list(range(num_classes)), fold=fold+1, save_folder=save_folder, dataset_type='Val')

        # Test evaluation
        test_acc, test_cm = evaluate_model(model, test_loader, device, num_classes)
        np.save(os.path.join(save_folder,f'test_cm_fold{fold+1}.npy'), test_cm)
        plot_confusion_matrix(test_cm, classes=list(range(num_classes)), fold=fold+1, save_folder=save_folder, dataset_type='Test')
        with open(results_file,'a') as f:
            f.write(f"Fold{fold+1} TestAcc={test_acc:.2f}%\n")
        print(f"Fold {fold+1} Test Accuracy: {test_acc:.2f}%")

        # 绘制训练曲线
        plot_curves(train_losses, val_losses, train_acc_list, val_acc_list, fold+1, save_folder)

        # 保存模型
        torch.save(model.state_dict(), os.path.join(save_folder,f'model_fold{fold+1}.pth'))

        val_scores.append(val_acc)
        test_scores.append(test_acc)

    # 总结
    print("\n=== Overall Summary ===")
    print(f"Val Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}")
    print(f"Test Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}")
    with open(results_file, 'a') as f:
        f.write(f"\n=== Overall Summary ===\nVal Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}\nTest Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}\n")
    
    print(f"\nAll results saved in {save_folder}")
    return save_folder


# ====================== 使用示例 ======================
# 假设 IQ 数据 shape=[num_samples, length, 2]，y=[num_samples]
#X_train_proc = preprocess_iq_dataset_led_rff(X_train)
#X_test_proc  = preprocess_iq_dataset_led_rff(X_test)
#y_train_torch = torch.tensor(y_train, dtype=torch.long)
#y_test_torch  = torch.tensor(y_test, dtype=torch.long)
#num_classes = len(np.unique(y_train_torch))
#save_folder = train_kfold(X_train_proc, y_train_torch, X_test_proc, y_test_torch, num_classes)

snr_list = list(range(20, -45, -5))  # 20, 15, 10, ..., -40

all_results = {}

for snr in snr_list:
    print("\n" + "="*60)
    print(f"🚀 开始实验：SNR = {snr} dB")
    print("="*60)

    # 覆盖全局变量
    SNR_DB = snr

    # 重新生成 RFF 特征（添加当前 SNR 噪声）
    X_train_proc = preprocess_iq_dataset_led_rff(
        X_train,
        snr_db=SNR_DB
    )
    X_test_proc = preprocess_iq_dataset_led_rff(
        X_test,
        snr_db=SNR_DB
    )

    # 转 tensor
    y_train_torch = torch.tensor(y_train, dtype=torch.long)
    y_test_torch = torch.tensor(y_test, dtype=torch.long)
    num_classes = len(np.unique(y_train_torch))

    # 训练并保存结果
    save_folder = train_kfold(
        X_train_proc, y_train_torch,
        X_test_proc,  y_test_torch,
        num_classes,
        snr_db=SNR_DB        # 传入当前 SNR
    )

    all_results[snr] = save_folder

print("\n===== 所有实验完成 =====")
for snr, folder in all_results.items():
    print(f"SNR={snr} dB 的结果保存在：{folder}")



c:\Users\ZY\.conda\envs\MW-RFF\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据集发射机数量： 6 具体为： ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
数据集接收机数量： 12 具体为： ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
数据集采集天数： 4 具体为： ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
6 12
✅ 训练样本数: 72000, 测试样本数: 72000
训练集所选日期： ['2021_03_15'] 测试集所选日期： ['2021_03_01']
X_train shape: (72000, 256, 2)
y_train shape: (72000,)
X_test  shape: (72000, 256, 2)
y_test  shape: (72000,)

🚀 开始实验：SNR = 20 dB


c:\Users\ZY\.conda\envs\MW-RFF\lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(



=== Fold 1/5 ===
Epoch 1/200 | TrainAcc=89.57% | ValAcc=74.73% | TrainLoss=0.3777 | ValLoss=0.7193 | AvgGrad=0.1133
Epoch 2/200 | TrainAcc=98.69% | ValAcc=69.17% | TrainLoss=0.0558 | ValLoss=0.8326 | AvgGrad=0.0936
Epoch 3/200 | TrainAcc=99.09% | ValAcc=77.62% | TrainLoss=0.0362 | ValLoss=0.6500 | AvgGrad=0.0777
Epoch 4/200 | TrainAcc=99.15% | ValAcc=91.55% | TrainLoss=0.0320 | ValLoss=0.2404 | AvgGrad=0.0739
Epoch 5/200 | TrainAcc=99.30% | ValAcc=85.15% | TrainLoss=0.0258 | ValLoss=0.4438 | AvgGrad=0.0642
Epoch 6/200 | TrainAcc=99.34% | ValAcc=98.03% | TrainLoss=0.0232 | ValLoss=0.0667 | AvgGrad=0.0622
Epoch 7/200 | TrainAcc=99.38% | ValAcc=89.81% | TrainLoss=0.0212 | ValLoss=0.2893 | AvgGrad=0.0591
Epoch 8/200 | TrainAcc=99.41% | ValAcc=83.94% | TrainLoss=0.0201 | ValLoss=0.6149 | AvgGrad=0.0570
Epoch 9/200 | TrainAcc=99.49% | ValAcc=94.24% | TrainLoss=0.0172 | ValLoss=0.1644 | AvgGrad=0.0528
Epoch 10/200 | TrainAcc=99.56% | ValAcc=98.67% | TrainLoss=0.0157 | ValLoss=0.0458 | AvgGra

c:\Users\ZY\.conda\envs\MW-RFF\lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(


...


=== Fold 1/5 ===
Epoch 1/200 | TrainAcc=16.80% | ValAcc=16.28% | TrainLoss=1.7943 | ValLoss=1.7946 | AvgGrad=0.0204
Epoch 2/200 | TrainAcc=17.09% | ValAcc=16.40% | TrainLoss=1.7927 | ValLoss=1.7989 | AvgGrad=0.0176
Epoch 3/200 | TrainAcc=17.46% | ValAcc=16.49% | TrainLoss=1.7916 | ValLoss=1.7949 | AvgGrad=0.0183
Epoch 4/200 | TrainAcc=18.10% | ValAcc=16.40% | TrainLoss=1.7902 | ValLoss=1.7985 | AvgGrad=0.0194
Epoch 5/200 | TrainAcc=18.35% | ValAcc=16.83% | TrainLoss=1.7889 | ValLoss=1.7934 | AvgGrad=0.0217
Epoch 6/200 | TrainAcc=18.89% | ValAcc=16.74% | TrainLoss=1.7872 | ValLoss=1.8013 | AvgGrad=0.0232
Epoch 7/200 | TrainAcc=19.82% | ValAcc=17.05% | TrainLoss=1.7842 | ValLoss=1.8136 | AvgGrad=0.0261
Epoch 8/200 | TrainAcc=20.25% | ValAcc=16.67% | TrainLoss=1.7818 | ValLoss=1.8132 | AvgGrad=0.0291
Epoch 9/200 | TrainAcc=20.89% | ValAcc=16.93% | TrainLoss=1.7776 | ValLoss=1.8151 | AvgGrad=0.0337
Epoch 10/200 | TrainAcc=21.73% | ValAcc=16.02% | TrainLoss=1.7727 | ValLoss=1.8307 | AvgGra

c:\Users\ZY\.conda\envs\MW-RFF\lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(



=== Fold 1/5 ===
Epoch 1/200 | TrainAcc=16.62% | ValAcc=16.38% | TrainLoss=1.7949 | ValLoss=1.8016 | AvgGrad=0.0210
Epoch 2/200 | TrainAcc=17.30% | ValAcc=16.69% | TrainLoss=1.7925 | ValLoss=1.8010 | AvgGrad=0.0186
Epoch 3/200 | TrainAcc=17.74% | ValAcc=17.36% | TrainLoss=1.7912 | ValLoss=1.7998 | AvgGrad=0.0183
Epoch 4/200 | TrainAcc=18.08% | ValAcc=16.44% | TrainLoss=1.7900 | ValLoss=1.7998 | AvgGrad=0.0198
Epoch 5/200 | TrainAcc=18.27% | ValAcc=16.41% | TrainLoss=1.7890 | ValLoss=1.7949 | AvgGrad=0.0211
Epoch 6/200 | TrainAcc=18.52% | ValAcc=16.80% | TrainLoss=1.7879 | ValLoss=1.7990 | AvgGrad=0.0233
Epoch 7/200 | TrainAcc=19.50% | ValAcc=16.35% | TrainLoss=1.7846 | ValLoss=1.8092 | AvgGrad=0.0252
Epoch 8/200 | TrainAcc=20.29% | ValAcc=16.76% | TrainLoss=1.7821 | ValLoss=1.8014 | AvgGrad=0.0282
Epoch 9/200 | TrainAcc=20.54% | ValAcc=16.89% | TrainLoss=1.7789 | ValLoss=1.8249 | AvgGrad=0.0319
Epoch 10/200 | TrainAcc=21.46% | ValAcc=16.80% | TrainLoss=1.7741 | ValLoss=1.8307 | AvgGra

c:\Users\ZY\.conda\envs\MW-RFF\lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(



=== Fold 1/5 ===
Epoch 1/200 | TrainAcc=16.74% | ValAcc=17.11% | TrainLoss=1.7945 | ValLoss=1.7927 | AvgGrad=0.0201
Epoch 2/200 | TrainAcc=17.02% | ValAcc=16.78% | TrainLoss=1.7930 | ValLoss=1.7949 | AvgGrad=0.0182
Epoch 3/200 | TrainAcc=17.54% | ValAcc=16.33% | TrainLoss=1.7914 | ValLoss=1.7990 | AvgGrad=0.0184
Epoch 4/200 | TrainAcc=17.82% | ValAcc=16.44% | TrainLoss=1.7907 | ValLoss=1.7963 | AvgGrad=0.0198
Epoch 5/200 | TrainAcc=18.63% | ValAcc=16.55% | TrainLoss=1.7885 | ValLoss=1.7993 | AvgGrad=0.0220
Epoch 6/200 | TrainAcc=19.26% | ValAcc=16.21% | TrainLoss=1.7864 | ValLoss=1.8191 | AvgGrad=0.0233
Epoch 7/200 | TrainAcc=19.60% | ValAcc=16.62% | TrainLoss=1.7844 | ValLoss=1.7985 | AvgGrad=0.0256
Epoch 8/200 | TrainAcc=20.32% | ValAcc=16.22% | TrainLoss=1.7815 | ValLoss=1.8074 | AvgGrad=0.0287
Epoch 9/200 | TrainAcc=20.82% | ValAcc=16.56% | TrainLoss=1.7780 | ValLoss=1.8110 | AvgGrad=0.0325
Early stopping.
Fold 1 Test Accuracy: 16.61%

=== Fold 2/5 ===
Epoch 1/200 | TrainAcc=16.48